In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType
from pyspark.sql.functions import *

In [0]:
display(df)

In [0]:
df = spark.read.csv('/Volumes/workspace/default/project_files/googleplaystore.csv', sep =',',  header=True, escape ='"', inferSchema=True)

In [0]:
df.count()

In [0]:
df.printSchema()

In [0]:
df = df.drop("Size", "Android Ver", "Last Updated", "Content Rating","Current Ver")

In [0]:
df.show(1)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import IntegerType, FloatType

df_clean = df.filter(~col("Reviews").contains("M"))
df_clean = df_clean.withColumn("Reviews", col("Reviews").cast(IntegerType())) \
                   .withColumn("Installs", regexp_replace(col("Installs"), "[^0-9]", "")) \
                   .withColumn("Installs", col("Installs").cast(IntegerType())) \
                   .withColumn("Price", regexp_replace(col("Price"), "[$]", "")) \
                   .withColumn("Price", col("Price").cast(FloatType()))
df_clean.createOrReplaceTempView("apps")
print("Cleanup complete!")

In [0]:
df_clean.printSchema()

In [0]:
df_clean.show(2)

In [0]:
df_clean.createOrReplaceTempView("apps")

In [0]:
%sql select * from apps

In [0]:
%sql SELECT App, SUM(Reviews) FROM apps
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10

In [0]:
%sql SELECT App, Type, SUM(Installs) FROM apps
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 10

In [0]:
%sql SELECT Category, SUM(Installs) FROM apps
GROUP BY 1
ORDER BY 2 DESC

In [0]:
%sql SELECT App, ROUND(SUM(Price), 2) FROM apps
WHERE Type = 'Paid'
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10

In [0]:
%sql SELECT Category, ROUND(AVG(Rating), 2) AS Average_Rating FROM apps
WHERE Rating IS NOT NULL AND Rating != 'NaN'
GROUP BY Category
HAVING COUNT(Reviews) > 10
ORDER BY Average_Rating DESC
LIMIT 10

In [0]:
%sql SELECT Type, ROUND(AVG(CAST(Rating AS FLOAT)), 2) AS Average_Rating, COUNT(*) AS Total_Apps FROM apps
WHERE Type IN ('Free', 'Paid') AND Rating IS NOT NULL AND Rating != 'NaN'
GROUP BY Type

In [0]:
%sql SELECT Genres, SUM(CAST(Installs AS BIGINT)) AS Total_Downloads FROM apps 
GROUP BY Genres
ORDER BY Total_Downloads DESC
LIMIT 10

In [0]:
%sql SELECT App, Category, ROUND((SUM(CAST(Reviews AS FLOAT)) / NULLIF(SUM(CAST(Installs AS FLOAT)), 0)) * 100, 2) AS Engagement_Percentage FROM apps
GROUP BY App, Category
HAVING SUM(CAST(Installs AS INT)) > 500000
ORDER BY Engagement_Percentage DESC
LIMIT 10